# NB4a: RadGraph-XL Entity-Relation Extraction
* **Notebook by:** Adam Lang
* **Date:** 4/2/2026
* **Purpose:** Zero-shot RadGraph-XL on veterinary radiology reports. Isolated environment
  with pinned `transformers<5` to avoid `BertTokenizer.encode_plus` incompatibility.

## What RadGraph Adds Over GliNER-BioMed
* GliNER-BioMed extracts **entities only** (anatomy, observation, disease, procedure)
* RadGraph extracts **entity-relation triples** (`located_at`, `suggestive_of`, modify)
* This directly addresses the relationship-level fabrication gap: all 6 radiologist-confirmed
  fabrication cases passed entity grounding (T6) with perfect scores because entities were real
  but relationships were fabricated

## Zero-Shot Transfer Hypothesis
- RadGraph was trained on human CXR reports (MIMIC-CXR). The entity types (Anatomy,
Observation) and relation types are domain-agnostic. The question is whether the model
**recognizes veterinary-specific anatomy and pathology terms.**
- RadGraph github: https://github.com/Stanford-AIMI/radgraph?tab=readme-ov-file#radgraph-xl
- RadGraph-XL paper: https://aclanthology.org/2024.findings-acl.765/


## Three-Phase Value
- You could use this framework in model development as follows:

1. **Before model training:** Build structured finding-to-diagnosis relationship graphs from expert reports
2. **During model training:** Entity-relation triples as structured output targets for SFT or DPO, etc..
3. * **Post training:** Relation-level grounding check -- are AI-asserted relationships real?

## Compute 
- Requires GPU cluster (`g5.48xlarge`). The notebook asserts transformers < 5 at startup so it fails fast if the install didn't work.

## 0. Install Dependencies (Pinned)

In [0]:
## 0. Critical: pin transformers<5 to avoid BertTokenizer.encode_plus removal
%pip install 'transformers<5' radgraph gliner torch --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Setup & Configuration

### 1.1 Imports

In [0]:
## 1.1 Imports
import json
import time
import gc
import pickle
import os
from typing import Dict, List, Tuple, Any
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch

from pyspark.sql import functions as F

# GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: {}".format(device))
if torch.cuda.is_available():
    print("GPU: {}".format(torch.cuda.get_device_name(0)))

import transformers
print("transformers version: {}".format(transformers.__version__))
assert int(transformers.__version__.split(".")[0]) < 5, "ERROR: transformers must be <5 for RadGraph"

print("Imports loaded.")

Device: cuda
GPU: NVIDIA A10G


2026-04-02 20:05:58.642309: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-02 20:05:58.662044: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-02 20:05:58.668009: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-02 20:05:58.683856: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-02 20:06:02.426994: W tensorflow/compiler/tf2

transformers version: 4.57.6
Imports loaded.


### 1.2 Configuration

In [0]:
## 1.2 Configuration

AB_TEST_BATCH = "mar_2026_batch_1"
AB_VARIANTS_TABLE = "llm_sandbox.gamuts.ab_test_variants"
AB_CASES_TABLE = "llm_sandbox.gamuts.ab_test_cases"

# GliNER labels (same as Track 2 for comparison)
GLINER_LABELS = [
    "anatomy definitely present", "anatomy definitely absent", "anatomy uncertain",
    "observation definitely present", "observation definitely absent", "observation uncertain",
    "disease definitely present", "disease definitely absent", "disease uncertain",
    "procedure definitely present", "procedure definitely absent", "procedure uncertain",
]
## NOTE: I tested this threshold on historical reports and it seemed to work well. 
## You can always adjust based on your data.
GLINER_THRESHOLD = 0.3

print("=" * 80)
print("NB4a: RADGRAPH-XL ENTITY-RELATION EXTRACTION")
print("=" * 80)
print("Batch: {}".format(AB_TEST_BATCH))
print("Device: {}".format(device))
print("=" * 80)

NB4a: RADGRAPH-XL ENTITY-RELATION EXTRACTION
Batch: mar_2026_batch_1
Device: cuda


## 2. Load Data

### 2.1 Load Round 3 AB Test Variants

In [0]:
## 2.1 Load Round 3 AB Test variants
variants_df = spark.sql('''
    SELECT
        v.case_id,
        v.variant_id,
        v.variant_conclusions,
        v.variant_recommendations,
        v.target_radiologist,
        c.anatomy_region,
        c.findings,
        c.history,
        c.signalment_context
    FROM {variants} v
    JOIN {cases} c ON v.case_id = c.case_id AND v.ab_test_batch = c.ab_test_batch
    WHERE v.ab_test_batch = '{batch}'
    AND v.case_id NOT LIKE 'DEMO_%'
    ORDER BY v.case_id, v.variant_id
'''.format(variants=AB_VARIANTS_TABLE, cases=AB_CASES_TABLE, batch=AB_TEST_BATCH))

variants_pd = variants_df.toPandas()
ai_df = variants_pd[variants_pd["variant_id"] != "A"].copy()
gt_df = variants_pd[variants_pd["variant_id"] == "A"].copy()

print("Loaded {} AI variants across {} cases".format(len(ai_df), ai_df["case_id"].nunique()))

Loaded 150 AI variants across 30 cases


## 3. Step 2a: GliNER-BioMed Entity Extraction (Baseline)

Run GliNER first as baseline for comparison, then release GPU for RadGraph.

### 3.1 Load GliNER-BioMed

In [0]:
## 3.1 Load GliNER-BioMed
from gliner import GLiNER

print("Loading GliNER-BioMed-large...")
gliner_model = GLiNER.from_pretrained("Ihor/gliner-biomed-large-v1.0")
gliner_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gliner_model = gliner_model.to(gliner_device)
gliner_model.eval()
print("GliNER-BioMed loaded on {}".format(gliner_device))


def parse_entity_label(raw_label: str) -> Tuple[str, str]:
    """Parse GliNER label into base type and polarity."""
    raw_label = raw_label.lower().strip()
    for polarity in ["definitely present", "definitely absent", "uncertain"]:
        if raw_label.endswith(polarity):
            base = raw_label.replace(polarity, "").strip()
            short_polarity = polarity.replace("definitely ", "")
            return base, short_polarity
    return raw_label, "unknown"


def extract_entities_gliner(text: str) -> List[dict]:
    """Extract entities from text using GliNER-BioMed."""
    if not text or len(text.strip()) < 10:
        return []
    try:
        entities_raw = gliner_model.predict_entities(text, GLINER_LABELS, threshold=GLINER_THRESHOLD)
        parsed = []
        for e in entities_raw:
            base_type, polarity = parse_entity_label(e["label"])
            parsed.append({
                "text": e["text"].lower().strip(),
                "label": e["label"],
                "base_type": base_type,
                "polarity": polarity,
                "score": float(e["score"]),
            })
        return parsed
    except Exception as ex:
        print("  GliNER error: {}".format(str(ex)[:100]))
        return []


print("GliNER extraction functions defined.")

Loading GliNER-BioMed-large...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-0bc77045-d4e8-4fb9-b115-10abd8d2e136/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

added_tokens.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

GliNER-BioMed loaded on cuda:0
GliNER extraction functions defined.


### 3.2 Run GliNER on ALL Variants

In [0]:
## 3.2 Run GliNER on all variants

print("=" * 80)
print("STEP 2a: GLINER-BIOMED ENTITY EXTRACTION (Baseline)")
print("=" * 80)

findings_entities = {}
claims_entities = {}

for idx, (_, row) in enumerate(ai_df.iterrows()):
    case_id = row["case_id"]
    variant_id = row["variant_id"]
    target_rad = row.get("target_radiologist", "") or ""
    findings = row.get("findings", "") or ""
    conc = row.get("variant_conclusions", "") or ""
    recs = row.get("variant_recommendations", "") or ""

    if case_id not in findings_entities:
        findings_entities[case_id] = extract_entities_gliner(findings)

    output_text = "{} {}".format(conc, recs)
    claims_entities[(case_id, variant_id, target_rad)] = extract_entities_gliner(output_text)

    if (idx + 1) % 30 == 0:
        print("  Progress: {}/{}".format(idx + 1, len(ai_df)))

# Release GliNER before loading RadGraph
del gliner_model
torch.cuda.empty_cache()
gc.collect()
print("\nGliNER released from GPU")

total_findings_ents = sum(len(v) for v in findings_entities.values())
total_output_ents = sum(len(v) for v in claims_entities.values())
print("\nGliNER Extraction Complete:")
print("  Findings entities: {} across {} cases".format(total_findings_ents, len(findings_entities)))
print("  AI output entities: {} across {} variants".format(total_output_ents, len(claims_entities)))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


STEP 2a: GLINER-BIOMED ENTITY EXTRACTION (Baseline)
  Progress: 30/150
  Progress: 60/150
  Progress: 90/150
  Progress: 120/150
  Progress: 150/150

GliNER released from GPU

GliNER Extraction Complete:
  Findings entities: 796 across 30 cases
  AI output entities: 4108 across 150 variants


## 4. Step 2b: RadGraph-XL Entity-Relation Extraction

### 4.1 Load RadGraph-XL model

In [0]:
## 4.1 Load RadGraph-XL
from radgraph import RadGraph

print("Loading RadGraph (modern-radgraph-xl)...")
try:
    radgraph_model = RadGraph(model_type="modern-radgraph-xl")
except TypeError:
    try:
        radgraph_model = RadGraph(model="modern-radgraph-xl")
    except TypeError:
        radgraph_model = RadGraph("modern-radgraph-xl")
print("RadGraph loaded.")

Loading RadGraph (modern-radgraph-xl)...
Using device: cuda:0


/databricks/python/lib/python3.12/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)


modern-radgraph-xl.tar.gz:   0%|          | 0.00/579M [00:00<?, ?B/s]

/usr/lib/python3.12/tarfile.py:2274: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`


RadGraph loaded.


### 4.2 RadGraph extraction function

In [0]:
## 4.2 RadGraph extraction function

def extract_radgraph(text: str) -> dict:
    """Extract entities and relations from text using RadGraph-XL.

    Returns dict with 'entities' list and 'relation_triples' list of
    (source_text, relation_type, target_text).
    """
    if not text or len(text.strip()) < 10:
        return {"entities": [], "relation_triples": []}

    try:
        annotations = radgraph_model([text])
        report_ann = annotations.get("0", {})
        entities_dict = report_ann.get("entities", {})

        entities = []
        entity_lookup = {}

        for eid, edata in entities_dict.items():
            token = edata.get("tokens", "")
            label = edata.get("label", "")

            entities.append({
                "entity_id": eid,
                "text": token.lower().strip(),
                "label": label,
            })
            entity_lookup[eid] = token.lower().strip()

        # Resolve relation triples
        relation_triples = []
        for eid, edata in entities_dict.items():
            source_text = edata.get("tokens", "").lower().strip()
            for rel in edata.get("relations", []):
                if len(rel) == 2:
                    rel_type, target_id = rel
                    target_text = entity_lookup.get(target_id, "UNKNOWN")
                    relation_triples.append({
                        "source": source_text,
                        "relation": rel_type,
                        "target": target_text,
                        "source_label": edata.get("label", ""),
                    })

        return {
            "entities": entities,
            "relation_triples": relation_triples,
        }

    except Exception as ex:
        print("  RadGraph error: {}".format(str(ex)[:100]))
        return {"entities": [], "relation_triples": []}


print("RadGraph extraction function defined.")

RadGraph extraction function defined.


### 4.3 Run RadGraph model on ALL variants

In [0]:
## 4.3 Run RadGraph on all variants

print("=" * 80)
print("STEP 2b: RADGRAPH-XL ENTITY-RELATION EXTRACTION (Zero-Shot Vet Radiology)")
print("=" * 80)

radgraph_findings = {}
radgraph_outputs = {}

for idx, (_, row) in enumerate(ai_df.iterrows()):
    case_id = row["case_id"]
    variant_id = row["variant_id"]
    target_rad = row.get("target_radiologist", "") or ""
    findings = row.get("findings", "") or ""
    conc = row.get("variant_conclusions", "") or ""
    recs = row.get("variant_recommendations", "") or ""

    if case_id not in radgraph_findings:
        radgraph_findings[case_id] = extract_radgraph(findings)

    output_text = "{} {}".format(conc, recs)
    radgraph_outputs[(case_id, variant_id, target_rad)] = extract_radgraph(output_text)

    if (idx + 1) % 10 == 0:
        print("  Progress: {}/{}".format(idx + 1, len(ai_df)))

# Cleanup
del radgraph_model
torch.cuda.empty_cache()
gc.collect()
print("\nRadGraph released from GPU")

# Summary stats
total_rg_findings_ents = sum(len(v["entities"]) for v in radgraph_findings.values())
total_rg_output_ents = sum(len(v["entities"]) for v in radgraph_outputs.values())
total_rg_findings_rels = sum(len(v["relation_triples"]) for v in radgraph_findings.values())
total_rg_output_rels = sum(len(v["relation_triples"]) for v in radgraph_outputs.values())

print("\nRadGraph Extraction Complete:")
print("  Findings: {} entities, {} relations across {} cases".format(
    total_rg_findings_ents, total_rg_findings_rels, len(radgraph_findings)))
print("  AI output: {} entities, {} relations across {} variants".format(
    total_rg_output_ents, total_rg_output_rels, len(radgraph_outputs)))

STEP 2b: RADGRAPH-XL ENTITY-RELATION EXTRACTION (Zero-Shot Vet Radiology)


/databricks/python/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:194: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


  Progress: 10/150
  Progress: 20/150
  Progress: 30/150
  Progress: 40/150
  Progress: 50/150
  Progress: 60/150
  Progress: 70/150
  Progress: 80/150
  Progress: 90/150
  Progress: 100/150
  Progress: 110/150
  Progress: 120/150
  Progress: 130/150
  Progress: 140/150
  Progress: 150/150

RadGraph released from GPU

RadGraph Extraction Complete:
  Findings: 1915 entities, 1069 relations across 30 cases
  AI output: 6248 entities, 3518 relations across 150 variants


## RadGraph-XL Zero-Shot Extraction Results

### GliNER-BioMed vs RadGraph-XL Comparison

| Metric | GliNER-BioMed | RadGraph-XL | RadGraph Advantage |
|--------|--------------|-------------|-------------------|
| Findings entities | 796 | 1,915 | 2.4x more entities |
| AI output entities | 4,108 | 6,248 | 1.5x more entities |
| Findings relations | 0 | **1,069** | Entirely new signal |
| AI output relations | 0 | **3,518** | Entirely new signal |

### Key Takeaway
- RadGraph extracts 2.4x more entities from findings AND provides 4,587 relation triples (1,069 findings + 3,518 AI output) that GliNER-BioMed cannot produce. 
- The zero-shot transfer from human CXR (MIMIC-CXR training data) to veterinary radiology is working — RadGraph recognizes vet anatomy and pathology terms well enough to extract structured entity-relation triples.

- **The 3,518 AI output relations are the new evaluation signal:** each one can be checked against the 1,069 findings relations to detect relationship-level fabrication that entity-only grounding (T6) misses.

## 5. Comparison: GliNER-BioMed vs RadGraph-XL

### 5.1 Side-by-side entity and relation counts

In [0]:
## 5.1 Side-by-side entity and relation counts

print("=" * 80)
print("GLINER-BIOMED vs RADGRAPH-XL COMPARISON")
print("=" * 80)

print("\n--- Entity Count Comparison ---")
print("{:<30} {:>15} {:>15}".format("", "GliNER-BioMed", "RadGraph-XL"))
print("-" * 60)
print("{:<30} {:>15} {:>15}".format("Findings entities", total_findings_ents, total_rg_findings_ents))
print("{:<30} {:>15} {:>15}".format("AI output entities", total_output_ents, total_rg_output_ents))
print("{:<30} {:>15} {:>15}".format("Findings relations", 0, total_rg_findings_rels))
print("{:<30} {:>15} {:>15}".format("AI output relations", 0, total_rg_output_rels))

if total_findings_ents > 0:
    print("\nRadGraph entity coverage vs GliNER:")
    print("  Findings: {:.1f}x".format(total_rg_findings_ents / total_findings_ents))
    print("  AI output: {:.1f}x".format(total_rg_output_ents / total_output_ents))

GLINER-BIOMED vs RADGRAPH-XL COMPARISON

--- Entity Count Comparison ---
                                 GliNER-BioMed     RadGraph-XL
------------------------------------------------------------
Findings entities                          796            1915
AI output entities                        4108            6248
Findings relations                           0            1069
AI output relations                          0            3518

RadGraph entity coverage vs GliNER:
  Findings: 2.4x
  AI output: 1.5x


### 5.2 Inspect RadGraph zero-shot transfer quality

In [0]:
## 5.2 Inspect RadGraph zero-shot transfer quality

print("=" * 80)
print("RADGRAPH ZERO-SHOT TRANSFER QUALITY INSPECTION")
print("=" * 80)

# Show 3 cases with entities and relations
shown = 0
for case_id, rg_result in radgraph_findings.items():
    if len(rg_result["entities"]) >= 3 and len(rg_result["relation_triples"]) >= 1:
        print("\n--- Case: {} ---".format(case_id))
        print("  Entities ({} total):".format(len(rg_result["entities"])))
        for e in rg_result["entities"][:10]:
            print("    '{}' [{}]".format(e["text"], e["label"]))
        if len(rg_result["entities"]) > 10:
            print("    ... +{} more".format(len(rg_result["entities"]) - 10))

        print("  Relations ({} total):".format(len(rg_result["relation_triples"])))
        for r in rg_result["relation_triples"][:10]:
            print("    {} --[{}]--> {}".format(r["source"], r["relation"], r["target"]))
        if len(rg_result["relation_triples"]) > 10:
            print("    ... +{} more".format(len(rg_result["relation_triples"]) - 10))

        shown += 1
        if shown >= 3:
            break

RADGRAPH ZERO-SHOT TRANSFER QUALITY INSPECTION

--- Case: 10181648 ---
  Entities (49 total):
    'faint' [Observation::definitely present]
    'soft tissue' [Observation::definitely present]
    'opacity' [Observation::definitely present]
    'protruding' [Observation::definitely present]
    'tracheal' [Anatomy::definitely present]
    'lumen' [Anatomy::definitely present]
    'cardiac' [Anatomy::definitely present]
    'silhouette' [Anatomy::definitely present]
    'loss' [Observation::definitely present]
    'caudal' [Anatomy::definitely present]
    ... +39 more
  Relations (31 total):
    faint --[modify]--> opacity
    protruding --[modify]--> opacity
    protruding --[located_at]--> lumen
    tracheal --[modify]--> lumen
    lumen --[modify]--> tracheal
    silhouette --[modify]--> cardiac
    caudal --[modify]--> cardiac
    waist --[located_at]--> cardiac
    elongated --[modify]--> projections
    elevation --[located_at]--> trachea
    ... +21 more

--- Case: 10911879 ---
 

#### Observations 
- The zero-shot transfer is working remarkably well. RadGraph correctly identifies:

1. **Vet anatomy:** stomach, intestinal tract, colon, bladder, kidneys, liver, acetabula, periarticular structures, tracheal lumen, vertebral bodies, disc space

2. **Observations with polarity:** "pathologic distention" as definitely absent, "osteophytosis" as definitely absent, "normal" as definitely present

3. **Clinically meaningful relations:** gas --[located_at]--> stomach, mineralized calculi --[located_at]--> bladder, narrowing --[located_at]--> disc space

--> The relations are a goldmine:
  - `located_at` ties findings to anatomy, modify captures severity/descriptors, and s`uggestive_of` (if present) would capture diagnostic reasoning. 
  - This is exactly the relational structure that the T6 function entity-only grounding couldn't capture.


### 5.3 Relation type distribution

In [0]:
## 5.3 Relation type distribution

print("=" * 80)
print("RELATION TYPE DISTRIBUTION")
print("=" * 80)

# Findings relations
findings_rel_types = []
for rg in radgraph_findings.values():
    for r in rg["relation_triples"]:
        findings_rel_types.append(r["relation"])

print("\nFindings relation types:")
for rel_type, count in Counter(findings_rel_types).most_common():
    print("  {}: {}".format(rel_type, count))

# AI output relations
output_rel_types = []
for rg in radgraph_outputs.values():
    for r in rg["relation_triples"]:
        output_rel_types.append(r["relation"])

print("\nAI output relation types:")
for rel_type, count in Counter(output_rel_types).most_common():
    print("  {}: {}".format(rel_type, count))

RELATION TYPE DISTRIBUTION

Findings relation types:
  modify: 544
  located_at: 516
  suggestive_of: 9

AI output relation types:
  modify: 1868
  located_at: 1321
  suggestive_of: 329


#### Observations 
- Very interesting distribution here: 


1. `suggestive_of` jumps from 9 in findings to 329 in AI output — that's the diagnostic reasoning signal. 
2. Findings describe what's observed (`located_at`, modify), while AI conclusions assert what findings mean (`suggestive_of`). 
3. Those 329 `suggestive_of` relations are exactly the diagnostic reasoning chains that we would want to validate against Gamuts.

### 5.4 Per-Variant relation extraction stats

In [0]:
## 5.4 Per-variant relation extraction stats

print("=" * 80)
print("PER-VARIANT RADGRAPH STATS")
print("=" * 80)

for vid in sorted(ai_df["variant_id"].unique()):
    variant_keys = [k for k in radgraph_outputs.keys() if k[1] == vid]
    ent_counts = [len(radgraph_outputs[k]["entities"]) for k in variant_keys]
    rel_counts = [len(radgraph_outputs[k]["relation_triples"]) for k in variant_keys]
    print("  {}: avg entities={:.1f}, avg relations={:.1f}, total rels={}".format(
        vid, np.mean(ent_counts), np.mean(rel_counts), sum(rel_counts)
    ))

PER-VARIANT RADGRAPH STATS
  B: avg entities=41.7, avg relations=22.9, total rels=688
  C: avg entities=42.5, avg relations=23.2, total rels=695
  D: avg entities=41.4, avg relations=23.7, total rels=2135


#### Observations 
- Per-case averages are nearly identical (22.9-23.7 relations) across variants — the AI produces similar relational density regardless of variant type. 
- Variant D's higher total is just 90 variants vs 30.

### 5.5 Entity label distribution comparison

In [0]:
## 5.5 Entity label distribution comparison

print("=" * 80)
print("ENTITY LABEL DISTRIBUTION")
print("=" * 80)

# GliNER base type distribution
gliner_types = []
for ents in claims_entities.values():
    for e in ents:
        gliner_types.append(e["base_type"])

print("\nGliNER-BioMed entity types (AI output):")
for etype, count in Counter(gliner_types).most_common():
    print("  {}: {}".format(etype, count))

# RadGraph entity label distribution
rg_types = []
for rg in radgraph_outputs.values():
    for e in rg["entities"]:
        rg_types.append(e["label"])

print("\nRadGraph entity labels (AI output):")
for etype, count in Counter(rg_types).most_common():
    print("  {}: {}".format(etype, count))

ENTITY LABEL DISTRIBUTION

GliNER-BioMed entity types (AI output):
  disease: 1647
  procedure: 975
  observation: 787
  anatomy: 699

RadGraph entity labels (AI output):
  Anatomy::definitely present: 2430
  Observation::definitely present: 1760
  Observation::uncertain: 1534
  Observation::definitely absent: 516
  Anatomy::uncertain: 8


#### Observations
- Interesting contrast noted: GliNER captures disease and procedure as separate types (2,622 combined) that RadGraph folds into Observation. RadGraph however captures polarity that GliNER handles differently — 1,534 `Observation::uncertain` is the hedging language critical for certainty calibration, and 516 `Observation::definitely` absent are negative findings. Complementary schemas.

## 6. Relationship-Level Fabrication Test
- **The core hypothesis:** cases where GliNER-BioMed entity grounding passes (entities are real)
but RadGraph relation extraction reveals fabricated relationships (entities are real
but their stated relationship is wrong).

### 6.1 Cross-case relation comparision: do AI output relations match findings relations?

In [0]:
## 6.1 Cross-case relation comparison: do AI output relations match findings relations?

print("=" * 80)
print("RELATIONSHIP-LEVEL FABRICATION DETECTION")
print("=" * 80)
print("For each AI variant, check if its entity-relations appear in the source findings.")

fabrication_results = []

for (case_id, variant_id, target_rad), output_rg in radgraph_outputs.items():
    findings_rg = radgraph_findings.get(case_id, {"relation_triples": []})

    # Build set of findings relations (normalized)
    findings_rels = set()
    for r in findings_rg["relation_triples"]:
        findings_rels.add((r["source"].lower(), r["relation"], r["target"].lower()))

    # Check each output relation
    grounded_rels = 0
    ungrounded_rels = 0
    ungrounded_examples = []

    for r in output_rg["relation_triples"]:
        rel_tuple = (r["source"].lower(), r["relation"], r["target"].lower())
        # Check exact match or reversed direction
        rev_tuple = (r["target"].lower(), r["relation"], r["source"].lower())

        if rel_tuple in findings_rels or rev_tuple in findings_rels:
            grounded_rels += 1
        else:
            ungrounded_rels += 1
            if len(ungrounded_examples) < 3:
                ungrounded_examples.append(rel_tuple)

    total_rels = grounded_rels + ungrounded_rels
    grounding_rate = grounded_rels / max(total_rels, 1)

    fabrication_results.append({
        "case_id": case_id,
        "variant_id": variant_id,
        "target_radiologist": target_rad,
        "total_output_rels": total_rels,
        "grounded_rels": grounded_rels,
        "ungrounded_rels": ungrounded_rels,
        "rel_grounding_rate": grounding_rate,
        "ungrounded_examples": ungrounded_examples,
    })

fab_df = pd.DataFrame(fabrication_results)

print("\n--- Per-Variant Relation Grounding ---")
for vid in sorted(fab_df["variant_id"].unique()):
    subset = fab_df[fab_df["variant_id"] == vid]
    mean_rate = subset["rel_grounding_rate"].mean()
    total_grounded = subset["grounded_rels"].sum()
    total_ungrounded = subset["ungrounded_rels"].sum()
    print("  {}: rel_grounding={:.1f}%, grounded={}, ungrounded={}".format(
        vid, 100 * mean_rate, total_grounded, total_ungrounded
    ))

print("\n--- Sample Ungrounded Relations (Potential Fabrications) ---")
for _, row in fab_df[fab_df["ungrounded_rels"] > 0].head(5).iterrows():
    print("  case={}, variant={}:".format(row["case_id"], row["variant_id"]))
    for src, rel, tgt in row["ungrounded_examples"]:
        print("    {} --[{}]--> {} (NOT in findings)".format(src, rel, tgt))

RELATIONSHIP-LEVEL FABRICATION DETECTION
For each AI variant, check if its entity-relations appear in the source findings.

--- Per-Variant Relation Grounding ---
  B: rel_grounding=9.7%, grounded=65, ungrounded=623
  C: rel_grounding=8.4%, grounded=67, ungrounded=628
  D: rel_grounding=9.1%, grounded=211, ungrounded=1924

--- Sample Ungrounded Relations (Potential Fabrications) ---
  case=10181648, variant=B:
    mild --[modify]--> cardiomegaly (NOT in findings)
    myxomatous --[modify]--> degeneration (NOT in findings)
    mitral --[modify]--> valve (NOT in findings)
  case=10181648, variant=C:
    mild --[modify]--> cardiomegaly (NOT in findings)
    grade 6 / 6 --[modify]--> systolic (NOT in findings)
    grade 6 / 6 --[modify]--> murmur (NOT in findings)
  case=10181648, variant=D:
    mild --[modify]--> cardiomegaly (NOT in findings)
    enlargement --[suggestive_of]--> degeneration (NOT in findings)
    myxomatous --[modify]--> degeneration (NOT in findings)
  case=10181648, va

/databricks/python/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


#### Important Observation
- **REALLY Important nuance before we interpret this:** `~9%` grounding is not a 91% fabrication rate. Most "ungrounded" relations are valid diagnostic reasoning that should differ from findings. 
- Look at the examples:
  - mild --[modify]--> cardiomegaly — this is a diagnosis derived from findings. The findings describe imaging signs; the conclusions name the condition. It's supposed to be new.
  - enlargement --[suggestive_of]--> degeneration — this IS the diagnostic reasoning chain. The AI is saying "this finding suggests this diagnosis." That's the job.
  - myxomatous --[modify]--> degeneration — diagnostic terminology not in imaging findings. Expected.

- The real fabrication signal is in `located_at` relations — if the AI says an observation is `located_at` an anatomy that the findings never mentioned, THAT's relationship-level fabrication. 
- So we need to run an refinement to further investigate this:

### 6.2 Refined Fabrication Detection

In [0]:
## 6.2 Refined fabrication detection: focus on located_at relations only
## suggestive_of and modify in conclusions are EXPECTED to differ from findings

print("=" * 80)
print("REFINED: LOCATED_AT RELATION GROUNDING (fabrication-sensitive)")
print("=" * 80)
print("Only checking located_at relations -- these should match findings.")
print("suggestive_of and modify in conclusions are expected diagnostic reasoning.\n")

refined_results = []

for (case_id, variant_id, target_rad), output_rg in radgraph_outputs.items():
    findings_rg = radgraph_findings.get(case_id, {"relation_triples": []})

    # Build findings located_at set
    findings_located = set()
    for r in findings_rg["relation_triples"]:
        if r["relation"] == "located_at":
            findings_located.add((r["source"].lower(), r["target"].lower()))
            findings_located.add((r["target"].lower(), r["source"].lower()))  # bidirectional

    # Also build findings entity set for partial matching
    findings_entities_set = set()
    for e in findings_rg.get("entities", []):
        findings_entities_set.add(e["text"].lower())

    # Check output located_at relations
    grounded = 0
    ungrounded = 0
    ungrounded_examples = []
    suggestive_of_count = 0

    for r in output_rg["relation_triples"]:
        if r["relation"] == "suggestive_of":
            suggestive_of_count += 1
            continue  # skip -- these are diagnostic reasoning, not factual claims
        if r["relation"] == "modify":
            continue  # skip -- modifier relations are descriptive, not locational

        # Only check located_at
        if r["relation"] == "located_at":
            pair = (r["source"].lower(), r["target"].lower())
            rev_pair = (r["target"].lower(), r["source"].lower())

            if pair in findings_located or rev_pair in findings_located:
                grounded += 1
            else:
                # Check if at least both entities appear in findings
                src_in = r["source"].lower() in findings_entities_set
                tgt_in = r["target"].lower() in findings_entities_set
                if src_in and tgt_in:
                    grounded += 1  # entities present, relationship plausible
                else:
                    ungrounded += 1
                    if len(ungrounded_examples) < 3:
                        ungrounded_examples.append({
                            "source": r["source"],
                            "target": r["target"],
                            "src_in_findings": src_in,
                            "tgt_in_findings": tgt_in,
                        })

    total = grounded + ungrounded
    refined_results.append({
        "case_id": case_id,
        "variant_id": variant_id,
        "located_at_total": total,
        "located_at_grounded": grounded,
        "located_at_ungrounded": ungrounded,
        "grounding_rate": grounded / max(total, 1),
        "suggestive_of_count": suggestive_of_count,
        "ungrounded_examples": ungrounded_examples,
    })

ref_df = pd.DataFrame(refined_results)

print("--- Per-Variant Located_At Grounding ---")
for vid in sorted(ref_df["variant_id"].unique()):
    subset = ref_df[ref_df["variant_id"] == vid]
    gr = subset["located_at_grounded"].sum()
    ungr = subset["located_at_ungrounded"].sum()
    total = gr + ungr
    sug = subset["suggestive_of_count"].sum()
    print("  {}: located_at_grounding={:.1f}% ({}/{}), suggestive_of={}".format(
        vid, 100 * gr / max(total, 1), gr, total, sug
    ))

print("\n--- Sample UNGROUNDED located_at (True Fabrication Candidates) ---")
for _, row in ref_df[ref_df["located_at_ungrounded"] > 0].head(10).iterrows():
    if row["ungrounded_examples"]:
        print("  case={}, variant={}:".format(row["case_id"], row["variant_id"]))
        for ex in row["ungrounded_examples"]:
            print("    {} --[located_at]--> {} (src_in_findings={}, tgt_in_findings={})".format(
                ex["source"], ex["target"], ex["src_in_findings"], ex["tgt_in_findings"]
            ))

REFINED: LOCATED_AT RELATION GROUNDING (fabrication-sensitive)
Only checking located_at relations -- these should match findings.
suggestive_of and modify in conclusions are expected diagnostic reasoning.

--- Per-Variant Located_At Grounding ---
  B: located_at_grounding=18.4% (47/255), suggestive_of=47
  C: located_at_grounding=16.7% (46/275), suggestive_of=54
  D: located_at_grounding=16.9% (134/791), suggestive_of=228

--- Sample UNGROUNDED located_at (True Fabrication Candidates) ---
  case=10181648, variant=B:
    degeneration --[located_at]--> valve (src_in_findings=False, tgt_in_findings=False)
    failure --[located_at]--> heart (src_in_findings=False, tgt_in_findings=False)
    redundant --[located_at]--> membrane (src_in_findings=False, tgt_in_findings=False)
  case=10181648, variant=C:
    degeneration --[located_at]--> valve (src_in_findings=False, tgt_in_findings=False)
    failure --[located_at]--> heart (src_in_findings=False, tgt_in_findings=False)
    mass --[located_

/databricks/python/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


## 7. Summary & Observations

### 7.1 Summary

In [0]:
## 7.1 Summary

print("=" * 100)
print("NB4a: RADGRAPH-XL ENTITY-RELATION EXTRACTION -- SUMMARY")
print("=" * 100)

print("\n1. ENTITY-RELATION EXTRACTION RESULTS")
print("   GliNER-BioMed: {} findings entities, {} output entities, 0 relations".format(
    total_findings_ents, total_output_ents))
print("   RadGraph-XL: {} findings entities, {} output entities".format(
    total_rg_findings_ents, total_rg_output_ents))
print("   RadGraph-XL: {} findings relations, {} output relations".format(
    total_rg_findings_rels, total_rg_output_rels))

print("\n2. ZERO-SHOT TRANSFER")
print("   RadGraph was trained on human CXR reports (MIMIC-CXR).")
print("   Applied zero-shot to veterinary radiology reports.")
print("   Entity coverage vs GliNER: {:.1f}x findings, {:.1f}x output".format(
    total_rg_findings_ents / max(total_findings_ents, 1),
    total_rg_output_ents / max(total_output_ents, 1),
))

print("\n3. RELATIONSHIP-LEVEL GROUNDING")
total_gr = fab_df["grounded_rels"].sum()
total_ungr = fab_df["ungrounded_rels"].sum()
overall_rate = total_gr / max(total_gr + total_ungr, 1)
print("   Overall relation grounding rate: {:.1f}%".format(100 * overall_rate))
print("   Grounded relations: {:,}".format(total_gr))
print("   Ungrounded relations: {:,}".format(total_ungr))

print("\n4. KEY FINDING")
print("   RadGraph provides entity-RELATION triples that GliNER-BioMed cannot.")
print("   Ungrounded relations = potential relationship-level fabrication.")
print("   This is the gap that entity-only grounding (T6) cannot detect.")

print("\n5. THREE-PHASE VALUE")
print("   Before training: Build relation graphs from expert reports for RAG curation")
print("   During training: Relation triples as structured output targets for SFT")
print("   After training: Relation-level grounding check in production pipeline")

print("\n" + "=" * 100)

NB4a: RADGRAPH-XL ENTITY-RELATION EXTRACTION -- SUMMARY

1. ENTITY-RELATION EXTRACTION RESULTS
   GliNER-BioMed: 796 findings entities, 4108 output entities, 0 relations
   RadGraph-XL: 1915 findings entities, 6248 output entities
   RadGraph-XL: 1069 findings relations, 3518 output relations

2. ZERO-SHOT TRANSFER
   RadGraph was trained on human CXR reports (MIMIC-CXR).
   Applied zero-shot to veterinary radiology reports.
   Entity coverage vs GliNER: 2.4x findings, 1.5x output

3. RELATIONSHIP-LEVEL GROUNDING
   Overall relation grounding rate: 9.7%
   Grounded relations: 343
   Ungrounded relations: 3,175

4. KEY FINDING
   RadGraph provides entity-RELATION triples that GliNER-BioMed cannot.
   Ungrounded relations = potential relationship-level fabrication.
   This is the gap that entity-only grounding (T6) cannot detect.

5. THREE-PHASE VALUE
   Before training: Build relation graphs from expert reports for RAG curation
   During training: Relation triples as structured output t

### 7.2 Observations

### Zero-Shot Transfer: Success
- RadGraph-XL trained on human CXR reports successfully extracts entities and relations from veterinary radiology reports. 1,915 findings entities (2.4x GliNER), 6,248 output entities (1.5x GliNER), and 4,587 relation triples (entirely new signal). 
- Vet anatomy terms (stomach, intestinal tract, acetabula, periarticular structures, vertebral bodies) are correctly recognized. More specialized vet terms (coxofemoral, costochondral) were not present in the 30 Round 3 cases but would need testing on musculoskeletal-heavy cases.

### Relation Type Distribution: Clinically Meaningful
- `modify` (544 findings / 1,868 output): descriptors and severity qualifiers
- `located_at` (516 findings / 1,321 output): observation-anatomy localization
- `suggestive_of` (9 findings / 329 output): diagnostic reasoning chains

The jump from 9 to 329 `suggestive_of` relations between findings and AI output IS the diagnostic reasoning signal. Findings describe; conclusions reason. **RadGraph captures this structural difference.**

### Fabrication Detection: Entity Linking Is the Bottleneck
- Raw relation grounding: `~9%` (91% "ungrounded") — misleading because `modify` and `suggestive_of` in conclusions are EXPECTED to differ from findings
- Refined `located_at` grounding: `~17%` — still misleading because diagnostic vocabulary differs from descriptive vocabulary
- Most "ungrounded" examples are valid clinical reasoning using different terminology ("degeneration `located_at` valve" vs "cardiac silhouette enlarged")
- **True fabrication detection requires semantic matching between diagnostic and descriptive vocabulary** — the same entity linking problem I previously identified in NB5a (Gamuts).

### Variant Comparison
- B, C, and D show nearly identical relation grounding rates (16.7-18.4%) and per-case relational density (~23 relations/variant). 
- Unlike Track 2 atomic verification where D had 3-6x more errors, RadGraph sees no structural difference between variants — the relation extraction is variant-agnostic, as expected for a syntax-level tool.

### Track 2 Cross-Reference
- Cannot directly compare RadGraph ungrounded relations against Track 2 CRIMSON fabrication labels because the "ungrounded" relations are dominated by the terminology gap, not true fabrication. 
- The 48 fabricated claims identified by CRIMSON in Track 2 (Variant D) are content-level errors that RadGraph's syntax-level extraction cannot isolate without semantic entity matching. 
- A proper cross-reference requires the entity linking bridge (HiT-SNOMED) to resolve vocabulary differences before comparing relation grounding against CRIMSON error labels.

### Architecture Validated, Bridge Missing
- RadGraph provides the RIGHT structure (entity-relation triples with typed relations). The evaluation framework is correct (check if output relations appear in findings). 
- The missing piece is the terminology bridge — mapping "mitral valve degeneration" to "enlarged cardiac silhouette with loss of caudal waist." 
- This is where HiT-SNOMED (NB4d) or BGE embedding similarity earns its value.

### Three-Phase Value (Confirmed)
1. **Before model training:** RadGraph on 66K historical corpus -- build structured finding-to-diagnosis relation graphs, identify common `suggestive_of` patterns per anatomy region

2. **During model training or fine-tuning:** Entity-relation triples as structured output targets for SFT or DPO -- train the model to produce claims with explicit relational grounding

3. **After training:** Relation-level grounding with semantic matching -- production-grade fabrication detection that T6 entity-only grounding cannot provide



### 7.3 Save Results Checkpoint

In [0]:
## 7.3 Save results checkpoint

checkpoint_nb4a = {
    "findings_entities": findings_entities,
    "claims_entities": claims_entities,
    "radgraph_findings": radgraph_findings,
    "radgraph_outputs": radgraph_outputs,
    "fabrication_results": fabrication_results,
}

with open("/dbfs/tmp/nb4a_radgraph_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint_nb4a, f)

print("NB4a checkpoint saved: {:.1f} MB".format(
    os.path.getsize("/dbfs/tmp/nb4a_radgraph_checkpoint.pkl") / 1e6))

NB4a checkpoint saved: 0.9 MB


---
## NB4a Complete. Proceed to NB4d (HiT-MiniLM-L12-SnomedCT) for ontology-aware entity linking.